# This is a notebook to read databento databases

In [18]:
import databento as db

In [19]:
# Read saved .dbn.zst
zst_file_path = r"../data/glbx-mdp3-20201020-20251019.ohlcv-1m.dbn.zst"
stored_data = db.DBNStore.from_file(zst_file_path)

# Convert to dataframe
df = stored_data.to_df()

In [20]:
print(df.tail(100))

                           rtype  publisher_id  instrument_id     open  \
ts_event                                                                 
2025-10-19 22:28:00+00:00     33             1         294973  6716.75   
2025-10-19 22:29:00+00:00     33             1         294973  6715.50   
2025-10-19 22:30:00+00:00     33             1         294973  6715.25   
2025-10-19 22:31:00+00:00     33             1         294973  6715.00   
2025-10-19 22:32:00+00:00     33             1         294973  6714.25   
...                          ...           ...            ...      ...   
2025-10-19 23:55:00+00:00     33             1         294973  6704.25   
2025-10-19 23:56:00+00:00     33             1         294973  6704.25   
2025-10-19 23:57:00+00:00     33             1         294973  6704.25   
2025-10-19 23:58:00+00:00     33             1         294973  6704.75   
2025-10-19 23:59:00+00:00     33             1         294973  6705.50   

                              high   

In [21]:
# 0) drop calendar spreads (ESZ1-ESH2, etc.)
df_processed = df[~df['symbol'].str.contains('-', na=False)].copy()

# припускаємо, що вже прибрали спреди та маєте index = ['ts_event','instrument_id']
tmp = df_processed.reset_index()
tmp['day'] = tmp['ts_event'].dt.tz_convert('UTC').dt.floor('D')

# 1) денні обсяги
daily_vol = (tmp.groupby(['day','instrument_id'])['volume']
               .sum()
               .reset_index())

# 2) фронт-інструмент на день (найбільший обсяг)
front = (daily_vol.sort_values(['day','volume'], ascending=[True, False])
                 .drop_duplicates(subset=['day'])
                 .rename(columns={'instrument_id':'front_instrument'})[['day','front_instrument']])

# 3) лишаємо на кожен день тільки фронт-місяць
picked = (tmp.merge(front, on='day', how='left')
            .loc[lambda x: x['instrument_id'] == x['front_instrument']]
            .drop(columns=['day','front_instrument'])
            .set_index('ts_event')
            .sort_index())

picked.to_csv(r"../data/glbx-mdp3-20201020-20251019.ohlcv-1m_higher_volume.csv")

/Users/yura/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev/_pydevd_bundle/pydevd_collect_try_except_info.py:153: DeprecationWarning: co_lnotab is deprecated, use co_lines instead.
  if not hasattr(co, 'co_lnotab'):
/Users/yura/Applications/PyCharm.app/Contents/plugins/python-ce/helpers/pydev/_pydevd_bundle/pydevd_collect_try_except_info.py:153: DeprecationWarning: co_lnotab is deprecated, use co_lines instead.
  if not hasattr(co, 'co_lnotab'):


In [9]:
# 0) drop calendar spreads (ESZ1-ESH2, etc.)
df_processed = df[~df['symbol'].str.contains('-', na=False)].copy()

# 1) make (ts_event, instrument_id) a MultiIndex
df_processed.index.name = 'ts_event'                  # name the index (optional but nice)

# 2) sort and drop duplicates on the index pair
df_processed = df_processed.sort_index()                        # sorts by ts_event then instrument_id
df_processed = df_processed[~df_processed.index.duplicated(keep='first')] # keep the earliest print per (ts_event, instrument)

# check
assert not df_processed.index.duplicated().any()

In [17]:
df_processed.to_csv(r"../data/glbx-mdp3-20201020-20251019.ohlcv-1m_source.csv")